# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SWAPI03/flyrank-ai-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**Among pages already ranking on Google's first page, which under-capture clicks for their position,
and therefore deserve a title/meta/snippet review first?** Lane 4 (CTR / Engagement Opportunity
Scoring). Unit = one content page; output = a ranked, reason-coded review queue; action = a human
edit; a wrong pick costs review time and edit risk. It is decision-support, not automation. This
notebook mirrors the deployed paper (`docs/index.html`).

In [1]:
import os, sys, json, subprocess, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git","clone","--depth","1",
            "https://github.com/flyrank-bih/flyrank-ml-internship-starter","flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/": os.chdir("..")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"dataset: {len(df):,} pages, {df['client_id'].nunique()} clients (anonymized slice of the ~79M-row release)")

dataset: 30,000 pages, 32 clients (anonymized slice of the ~79M-row release)


## 2. Data

FlyRank ML Internship release (full warehouse ~79M daily rows, 104 clients, build `v20260703`); this
notebook runs on the shipped anonymized slice, 90-day aggregates. **Excluded on purpose:**
`trend_direction`/`trend_pct` (label-derived), `clicks_90d` as a feature (reconstructs CTR), product
flags (not shipped), and any raw URL/query/title/client name. IDs are for grouping only.

In [2]:
vis = df[(df["impressions_90d"]>=100) & (df["ctr"].notna())].copy()
vis["tier_median_ctr"] = vis.groupby("position_tier")["ctr"].transform("median")
d = df[(df["impressions_90d"]>=500) & (df["avg_position"]>0) & (df["avg_position"]<=20) & (df["ctr"]>0)].copy()
d["log_impressions"] = np.log1p(d["impressions_90d"])
NUM=["avg_position","log_impressions","content_age_days","days_since_last_update","word_count","search_volume","competition","cpc"]
CAT=["content_type","main_intent"]
for c in NUM: d[c]=pd.to_numeric(d[c],errors="coerce")
d[NUM]=d[NUM].fillna(d[NUM].median()); d[CAT]=d[CAT].fillna("unknown")
d["ctr_c"]=d["ctr"].clip(upper=d["ctr"].quantile(0.99))
print(f"visible pool: {len(vis):,} | review population (visible, pos 1-20, ctr>0): {len(d):,}")

visible pool: 22,006 | review population (visible, pos 1-20, ctr>0): 10,807


## 3. Methodology

Expected-CTR regression ranked by residual. Features are pre-click observables (position, log
impressions, age, freshness, word count, search volume, competition, CPC, content type, intent);
target is clipped CTR; baseline is the position-tier median; validation is a client-grouped split
(seed 42). Leakage check: adding `clicks_90d` spikes R² to ~0.98, so it stays out.

In [3]:
pre = lambda: ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore"), CAT)], remainder="passthrough")
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr, te = next(gss.split(d, d["ctr_c"], d["client_id"]))
tmed = d.iloc[tr].groupby("position_tier")["ctr_c"].median(); gmed = d.iloc[tr]["ctr_c"].median()
yte = d.iloc[te]["ctr_c"]; base = d.iloc[te]["position_tier"].map(tmed).fillna(gmed)
rows = [("Baseline (tier median)", mean_absolute_error(yte,base), r2_score(yte,base))]
for name,mdl in [("Linear Regression",LinearRegression()),
                 ("Decision Tree (d=5)",DecisionTreeRegressor(max_depth=5,random_state=42)),
                 ("Gradient Boosting",GradientBoostingRegressor(random_state=42))]:
    p=Pipeline([("pre",pre()),("m",mdl)]).fit(d.iloc[tr][NUM+CAT], d.iloc[tr]["ctr_c"])
    pr=p.predict(d.iloc[te][NUM+CAT]); rows.append((name, mean_absolute_error(yte,pr), r2_score(yte,pr)))
tbl=pd.DataFrame(rows, columns=["method","MAE","R2"]).set_index("method").round(4)
# leakage confession
d["clk_f"]=pd.to_numeric(d["clicks_90d"],errors="coerce").fillna(0)
pl=Pipeline([("pre",ColumnTransformer([("c",OneHotEncoder(handle_unknown="ignore"),CAT)],remainder="passthrough")),
            ("m",GradientBoostingRegressor(random_state=42))]).fit(d.iloc[tr][NUM+CAT+["clk_f"]], d.iloc[tr]["ctr_c"])
r2_leak=r2_score(yte, pl.predict(d.iloc[te][NUM+CAT+["clk_f"]]))
print("=== 4. Results: model vs baseline (client-holdout) ==="); print(tbl.to_string())
print(f"\nleakage confession: +clicks_90d -> R2 {r2_leak:.3f} (rejected)")

=== 4. Results: model vs baseline (client-holdout) ===
                           MAE      R2
method                                
Baseline (tier median)  0.2966 -0.2423
Linear Regression       0.2917 -0.0337
Decision Tree (d=5)     0.2843 -0.0506
Gradient Boosting       0.2864  0.0013

leakage confession: +clicks_90d -> R2 0.978 (rejected)


## 4. Results (vs baseline)

The learned models beat the tier-median baseline modestly on absolute error and clearly on R², but a
depth-5 tree ties gradient boosting on MAE, so complexity is not rewarded. The headline finding is a
validation one (below): a random split flatters the model; the honest client-grouped split reveals
the truth.

In [4]:
idx=np.arange(len(d)); tr_r,te_r=train_test_split(idx,test_size=0.25,random_state=42)
def gbm_r2(a,b):
    p=Pipeline([("pre",pre()),("m",GradientBoostingRegressor(random_state=42))]).fit(d.iloc[a][NUM+CAT], d.iloc[a]["ctr_c"])
    return r2_score(d.iloc[b]["ctr_c"], p.predict(d.iloc[b][NUM+CAT]))
r2_random=gbm_r2(tr_r,te_r); r2_group=float(tbl.loc["Gradient Boosting","R2"])
print(f"honest split, R2:  random (before) {r2_random:.3f}  ->  client-grouped (after) {r2_group:.3f}")
print("Observed: the drop is memorization the random split hid. On unseen clients the model is only")
print("directional -- so the shippable scorer stays the transparent tier-median rule.")

honest split, R2:  random (before) 0.143  ->  client-grouped (after) 0.001
Observed: the drop is memorization the random split hid. On unseen clients the model is only
directional -- so the shippable scorer stays the transparent tier-median rule.


## 5. Limitations

Observational, one 90-day snapshot; surfaces pages that *look* worth reviewing but cannot show a title
edit *causes* clicks (needs an experiment). Feature and target share the window (scoring, not
prediction). Small held-out client count → directional. No algorithm/AI-ranking claims; no
client-identifying output.

In [5]:
vis["gap"]=(vis["tier_median_ctr"]-vis["ctr"]).clip(lower=0)
def arch(r):
    if r["ctr"]>=r["tier_median_ctr"]: return "healthy_ctr"
    if r["impressions_90d"]<500: return "low_volume_watch"
    if r["avg_position"]>7 and r["position_tier"]=="page_1": return "bottom_of_page_one"
    if r["impressions_90d"]>=3000 and r["ctr"]<0.5*r["tier_median_ctr"]: return "high_value_ctr_gap"
    if r["ctr"]<0.5*r["tier_median_ctr"]: return "ctr_gap_candidate"
    return "minor_ctr_gap"
vis["archetype"]=vis.apply(arch,axis=1)
queue=vis[vis["archetype"].isin(["high_value_ctr_gap","ctr_gap_candidate"])].copy()
queue["score"]=queue["gap"]*np.log1p(queue["impressions_90d"])
queue["in_decay"]=(queue["content_age_days"]>=271)&(queue["content_age_days"]<=365)
queue=queue.sort_values("score",ascending=False)
print(f"action queue: {len(queue):,} pages | decay-window overlap: {int(queue['in_decay'].sum())}")
print(f"exposure at stake in top 50: {int(queue.head(50)['impressions_90d'].sum()):,} impressions/90d")

action queue: 3,237 pages | decay-window overlap: 659
exposure at stake in top 50: 2,015,419 impressions/90d


## 6. Ranked recommendations

A human-reviewed action queue: `high_value_ctr_gap` → priority review, `ctr_gap_candidate` → review;
`bottom_of_page_one` and `low_volume_watch` are held out (monitor only). No auto-editing; zero-click
high-impression pages get a mandatory human check; no causal claims; pseudonymized IDs only. This is
the paper's recommendations section.

In [6]:
# 7. Artifacts the paper embeds: regenerate the figures + a metrics receipt.
os.makedirs("work/figures", exist_ok=True); os.makedirs("work/outputs", exist_ok=True)
order=["top_3","striking","page_1","page_3_5","deep"]
tm=vis.groupby("position_tier")["ctr"].mean().reindex([o for o in order if o in vis["position_tier"].unique()])
fig,ax=plt.subplots(figsize=(6.4,3.4)); tm.plot(kind="bar",ax=ax,color="#c0392b")
ax.set_title("Mean CTR by position tier"); ax.set_ylabel("CTR (x100)"); plt.xticks(rotation=0)
fig.savefig("work/figures/paper_ctr_by_tier.png",dpi=120,bbox_inches="tight"); plt.close(fig)
fig,ax=plt.subplots(figsize=(6.4,3.4))
ax.bar(["random","client-grouped"],[r2_random,r2_group],color=["#7f8c8d","#2980b9"])
ax.set_title("Expected-CTR R2: random vs honest split"); ax.set_ylabel("R2")
fig.savefig("work/figures/paper_honest_split.png",dpi=120,bbox_inches="tight"); plt.close(fig)
json.dump({"review_population":int(len(d)),"model_vs_baseline":tbl.reset_index().to_dict("records"),
           "r2_random":round(r2_random,3),"r2_group":round(r2_group,3),"leak_r2":round(r2_leak,3),
           "queue_size":int(len(queue)),"decay_overlap":int(queue["in_decay"].sum())},
          open("work/outputs/capstone_metrics.json","w"), indent=2)
print("artifacts written: work/figures/paper_*.png, work/outputs/capstone_metrics.json")
print("Deployed paper: docs/index.html  ->  GitHub Pages (/docs).")

artifacts written: work/figures/paper_*.png, work/outputs/capstone_metrics.json
Deployed paper: docs/index.html  ->  GitHub Pages (/docs).


## 7. Artifacts the paper embeds

The figures and metrics above are what the deployed page shows. The paper itself is
`docs/index.html`, served via GitHub Pages from `/docs`; its exact URL is recorded in
`submission/paper_url.txt`.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
